# Factor 1 — GC Content / GC 含量差異

**Owner / 負責人:** Alex Chen  
**Step:** 03 Feature Weighting → Factor 1  
**Expected runtime / 預計執行時間:** ~5 min first run (host genome download), <30 s cached / 首次約 5 分鐘（需下載宿主基因組），後續快取後 30 秒內完成

---

## Biological rationale / 生物學原理

GC content reflects long-term co-evolution between a phage and its host. When a phage infects a host for many generations, selection pressure tends to align their genomic GC% so that phage genes are translated efficiently by the host's tRNA pool. A smaller `|GC_phage − GC_host|` therefore correlates with higher infection likelihood.

GC 含量反映噬菌體與宿主之間的長期共演化關係。當噬菌體長期感染某一宿主時，天擇壓力會使兩者的 GC% 趨向一致，以確保噬菌體基因能被宿主 tRNA 池高效翻譯。因此，`|GC噬菌體 − GC宿主|` 越小，感染可能性越高。

**Feature definition / 特徵定義:**  
`x = |GC%_phage − GC%_host|` (whole-genome, fraction 0–1)

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import ssl, os, subprocess, zipfile, tempfile, pathlib, time, warnings

# Bypass corporate SSL inspection if needed
ssl._create_default_https_context = ssl._create_unverified_context

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from Bio import SeqIO, Entrez
from Bio.SeqUtils import gc_fraction

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
Entrez.email = 'alexchenworking1.618@gmail.com'
print('Imports OK')

Imports OK


In [ ]:
# ── Config — edit paths here if your layout differs ─────────────────────────
# To use the updated truth table, change TRUTH_TABLE_PATH below.
# 更換真值表時，只需修改 TRUTH_TABLE_PATH 這一行。

REPO_ROOT = pathlib.Path().resolve()
while not (REPO_ROOT / 'CLAUDE.md').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

TRUTH_TABLE_PATH = REPO_ROOT / '02_annotation/outputs/host_proteins/pairs.csv'
# ↑ Swap to combined_matrix after reshaping, or to any long-format CSV with columns: phage_id, host_id, y

MANIFEST_PATH    = REPO_ROOT / '02_annotation/outputs/host_proteins/manifest.csv'
PHAGE_RUNS_DIR   = REPO_ROOT / '02_annotation/outputs/pharokka_runs'
HOST_GENOMES_DIR = REPO_ROOT / '01_data_ground_truth/outputs/host_genomes/by_organism'
OUT_DIR          = REPO_ROOT / '03_feature_weighting/outputs/per_factor/factor1'

HOST_GENOMES_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT      : {REPO_ROOT}')
for name, p in [('TRUTH_TABLE', TRUTH_TABLE_PATH), ('MANIFEST', MANIFEST_PATH),
                ('PHAGE_RUNS_DIR', PHAGE_RUNS_DIR), ('HOST_GENOMES', HOST_GENOMES_DIR), ('OUT_DIR', OUT_DIR)]:
    print(f'{name:15s}: .../{p.relative_to(REPO_ROOT)}')

REPO_ROOT      : /Users/alexy/Desktop/Claude Workspace/iGEM_Claremont_2026
TRUTH_TABLE    : .../02_annotation/outputs/host_proteins/pairs.csv
MANIFEST       : .../02_annotation/outputs/host_proteins/manifest.csv
PHAGE_RUNS_DIR : .../02_annotation/outputs/pharokka_runs
HOST_GENOMES   : .../01_data_ground_truth/outputs/host_genomes/by_organism
OUT_DIR        : .../03_feature_weighting/outputs/per_factor/factor1


In [ ]:
# ── Cell 3: Acquire host genomes ─────────────────────────────────────────────
# Strategy: try ncbi-datasets CLI first; fall back to Bio.Entrez.
# Idempotent — skips hosts already on disk.
# 冪等操作：若 .fna 已存在則跳過。優先使用 datasets CLI，失敗時改用 Bio.Entrez。

manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest: {len(manifest)} hosts')
print(manifest[['host_id','kind','assembly','organism']].to_string(index=False))
print()

def fetch_host_genome(host_id: str, gcf: str, out_dir: pathlib.Path) -> pathlib.Path:
    dest_dir = out_dir / host_id
    dest_dir.mkdir(parents=True, exist_ok=True)
    existing = list(dest_dir.glob('*.fna'))
    if existing:
        print(f'  cached  {host_id}')
        return existing[0]

    # ── Try datasets CLI ────────────────────────────────────────────────────
    print(f'  fetching {gcf} ({host_id}) via datasets CLI...', end=' ', flush=True)
    try:
        with tempfile.TemporaryDirectory() as tmp:
            zip_path = pathlib.Path(tmp) / 'dataset.zip'
            r = subprocess.run(['datasets','download','genome','accession', gcf,
                                '--include','genome','--filename', str(zip_path)],
                               capture_output=True, text=True, timeout=120)
            if r.returncode == 0:
                with zipfile.ZipFile(zip_path) as zf:
                    fna_names = [n for n in zf.namelist() if n.endswith('.fna')]
                    fna_path = dest_dir / f'{host_id}.fna'
                    with zf.open(fna_names[0]) as src, open(fna_path,'wb') as dst:
                        dst.write(src.read())
                print('OK (datasets CLI)')
                return fna_path
    except Exception:
        pass

    # ── Fallback: Bio.Entrez ─────────────────────────────────────────────────
    print('falling back to Entrez...', end=' ', flush=True)
    handle = Entrez.esearch(db='assembly', term=gcf)
    asm_rec = Entrez.read(handle); handle.close()
    asm_id = asm_rec['IdList'][0]
    # Try RefSeq link first, then INSDC
    nuc_ids = []
    for lname in ['assembly_nuccore_refseq', 'assembly_nuccore_insdc']:
        handle = Entrez.elink(dbfrom='assembly', db='nuccore', id=asm_id, linkname=lname)
        links = Entrez.read(handle); handle.close()
        nuc_ids = [x['Id'] for x in links[0]['LinkSetDb'][0]['Link']] if links[0].get('LinkSetDb') else []
        if nuc_ids:
            break
    fna_path = dest_dir / f'{host_id}.fna'
    handle = Entrez.efetch(db='nuccore', id=','.join(nuc_ids), rettype='fasta', retmode='text')
    with open(fna_path, 'w') as fh:
        fh.write(handle.read())
    handle.close()
    time.sleep(0.4)
    print('OK (Entrez)')
    return fna_path

host_fna_paths = {}
for _, row in manifest.iterrows():
    try:
        fna = fetch_host_genome(row['host_id'], row['assembly'], HOST_GENOMES_DIR)
        host_fna_paths[row['host_id']] = fna
    except Exception as e:
        print(f'  FAIL {row["host_id"]}: {e}')

print(f'\n{len(host_fna_paths)}/{len(manifest)} host genomes ready.')

Manifest: 10 hosts
                       host_id kind        assembly                                         organism
         Xanthomonas_citri_306 host GCF_000007165.1             Xanthomonas citri pv. citri str. 306
  Xanthomonas_campestris_33913 host GCF_000007145.1 Xanthomonas campestris pv. campestris ATCC 33913
  Xanthomonas_oryzae_KACC10331 host GCF_000007385.1         Xanthomonas oryzae pv. oryzae KACC 10331
   Escherichia_coli_K12_MG1655 host GCF_000005845.2                     Escherichia coli K-12 MG1655
         Bacillus_subtilis_168  neg GCF_000009045.1                            Bacillus subtilis 168
   Pseudomonas_aeruginosa_PAO1  neg GCF_000006765.1                      Pseudomonas aeruginosa PAO1
Staphylococcus_aureus_NCTC8325  neg GCF_000013425.1                  Staphylococcus aureus NCTC 8325
Mycobacterium_smegmatis_MC2155  neg GCF_000015005.1                  Mycobacterium smegmatis MC2 155
       Salmonella_enterica_LT2  neg GCF_000006945.2                     

In [ ]:
# ── Cell 4: Load phage DNA from pharokka .gbk ────────────────────────────────
# Concatenates all contigs per phage into a single sequence for GC computation.

pairs = pd.read_csv(TRUTH_TABLE_PATH)
phage_ids = pairs['phage_id'].unique()
print(f'Truth table: {len(pairs)} rows, {len(phage_ids)} unique phages')

phage_records = {}
for phage_id in phage_ids:
    gbk_path = PHAGE_RUNS_DIR / phage_id / 'pharokka.gbk'
    if not gbk_path.exists():
        print(f'  ✗ Missing: {gbk_path}'); continue
    contigs = list(SeqIO.parse(gbk_path, 'genbank'))
    concat_seq = ''.join(str(r.seq) for r in contigs)
    phage_records[phage_id] = {'seq': concat_seq, 'length': len(concat_seq)}
    print(f'  ✓ {phage_id:15s}  {len(contigs)} contig(s)  {len(concat_seq):,} bp')

print(f'\n{len(phage_records)}/{len(phage_ids)} phage genomes loaded.')

Truth table: 50 rows, 5 unique phages
  ✓ AB720063.2       1 contig(s)  43,870 bp
  ✓ AB720064.1       1 contig(s)  42,963 bp
  ✓ AP008979.1       1 contig(s)  43,785 bp
  ✓ EU717894.1       1 contig(s)  44,080 bp
  ✓ JN882298.1       1 contig(s)  42,608 bp

5/5 phage genomes loaded.


In [ ]:
# ── Cell 5: Compute per-genome GC% ───────────────────────────────────────────
# gc_fraction() returns float 0-1. Cross-checks phage values against pharokka's
# pre-computed GC (pharokka_length_gc_cds_density.tsv). Flags |delta| > 0.005.

gc_rows = []
print(f"{'genome':20s}  {'gc_ours':>8s}  {'gc_pharokka':>11s}  {'delta':>6s}  flag")

for phage_id, rec in phage_records.items():
    gc = gc_fraction(rec['seq'])
    gc_rows.append({'genome_id': phage_id, 'kind': 'phage', 'gc_whole': round(gc,6), 'length_bp': rec['length']})
    tsv = PHAGE_RUNS_DIR / phage_id / 'pharokka_length_gc_cds_density.tsv'
    if tsv.exists():
        pk_gc = pd.read_csv(tsv, sep='\t')['gc_perc'].iloc[0]
        d = abs(gc - pk_gc)
        print(f'  {phage_id:18s}  {gc:8.4f}  {pk_gc:11.4f}  {d:6.4f}  {"⚠" if d>0.005 else "✓"}')

kind_map = manifest.set_index('host_id')['kind'].to_dict()
print('\nHost GC:')
for host_id, fna_path in host_fna_paths.items():
    contigs = list(SeqIO.parse(fna_path, 'fasta'))
    seq = ''.join(str(r.seq) for r in contigs)
    gc = gc_fraction(seq)
    gc_rows.append({'genome_id': host_id, 'kind': kind_map.get(host_id,'host'), 'gc_whole': round(gc,6), 'length_bp': len(seq)})
    print(f'  {host_id:45s}  gc={gc:.4f}  ({len(seq):,} bp)  [{kind_map.get(host_id,"host")}]')

gc_df = pd.DataFrame(gc_rows)
gc_df.to_csv(OUT_DIR / 'per_genome_gc.csv', index=False)
print(f'\n✓ Saved per_genome_gc.csv\n')
print(gc_df.to_string(index=False))

genome                 gc_ours  gc_pharokka   delta  flag
  AB720063.2            0.5331       0.5300  0.0031  ✓
  AB720064.1            0.6702       0.6700  0.0002  ✓
  AP008979.1            0.5108       0.5100  0.0008  ✓
  EU717894.1            0.5564       0.5600  0.0036  ✓
  JN882298.1            0.5159       0.5200  0.0041  ✓

Host GC:
  Xanthomonas_citri_306                          gc=0.6471  (5,274,174 bp)  [host]
  Xanthomonas_campestris_33913                   gc=0.6507  (5,076,188 bp)  [host]
  Xanthomonas_oryzae_KACC10331                   gc=0.6369  (4,941,439 bp)  [host]
  Escherichia_coli_K12_MG1655                    gc=0.5079  (4,641,652 bp)  [host]
  Bacillus_subtilis_168                          gc=0.4351  (4,215,606 bp)  [neg]
  Pseudomonas_aeruginosa_PAO1                    gc=0.6656  (6,264,404 bp)  [neg]
  Staphylococcus_aureus_NCTC8325                 gc=0.3287  (2,821,361 bp)  [neg]
  Mycobacterium_smegmatis_MC2155                 gc=0.6740  (6,988,209 bp)  [ne

In [ ]:
# ── Cell 6 & 7: Build per-pair feature + Z-score normalize ───────────────────
# x_gc = |GC_phage − GC_host|  then Z-score normalized.

gc_lkp  = gc_df.set_index('genome_id')['gc_whole'].to_dict()
knd_lkp = gc_df.set_index('genome_id')['kind'].to_dict()

feat_rows = []
for _, row in pairs.iterrows():
    p, h, y = row['phage_id'], row['host_id'], row['y']
    gc_p, gc_h = gc_lkp.get(p), gc_lkp.get(h)
    if gc_p is None or gc_h is None:
        print(f'  ✗ missing GC for ({p},{h})'); continue
    feat_rows.append({'phage_id': p, 'host_id': h, 'y': y,
                      'host_kind': knd_lkp.get(h, 'unknown'),
                      'gc_phage': round(gc_p, 6), 'gc_host': round(gc_h, 6),
                      'x_gc': round(abs(gc_p - gc_h), 6)})

feat_df = pd.DataFrame(feat_rows)
mu, sigma = feat_df['x_gc'].mean(), feat_df['x_gc'].std()
feat_df['x_gc_zscore'] = ((feat_df['x_gc'] - mu) / sigma).round(6)

feat_df.to_csv(OUT_DIR / 'f01_gc_content.csv', index=False)
print(f'✓ Saved f01_gc_content.csv  ({len(feat_df)} rows  μ={mu:.4f}  σ={sigma:.4f})')
print()
print(feat_df.head(20).to_string(index=False))
print('...')
print(f'(50 rows total — see f01_gc_content.csv for full table)')

✓ Saved f01_gc_content.csv  (50 rows  μ=0.1045  σ=0.0724)

  phage_id                        host_id  y host_kind  gc_phage  gc_host     x_gc  x_gc_zscore
AB720063.2          Xanthomonas_citri_306  1      host  0.533098 0.647117 0.114019     0.131809
AB720063.2   Xanthomonas_campestris_33913  0      host  0.533098 0.650686 0.117588     0.181098
AB720063.2   Xanthomonas_oryzae_KACC10331  0      host  0.533098 0.636933 0.103835    -0.008836
AB720063.2    Escherichia_coli_K12_MG1655  0      host  0.533098 0.507907 0.025191    -1.094942
AB720063.2          Bacillus_subtilis_168  0       neg  0.533098 0.435144 0.097954    -0.090055
AB720063.2    Pseudomonas_aeruginosa_PAO1  0       neg  0.533098 0.665557 0.132459     0.386473
AB720063.2 Staphylococcus_aureus_NCTC8325  0       neg  0.533098 0.328683 0.204415     1.380215
AB720063.2 Mycobacterium_smegmatis_MC2155  0       neg  0.533098 0.674028 0.140930     0.503461
AB720063.2        Salmonella_enterica_LT2  0       neg  0.533098 0.522389 0.0

In [ ]:
# ── Cell 8: Correlation stats + plots ────────────────────────────────────────

from scipy import stats as scipy_stats
pr, pp = scipy_stats.pearsonr(feat_df['x_gc'], feat_df['y'])
sr, sp = scipy_stats.spearmanr(feat_df['x_gc'], feat_df['y'])

print('── Correlation: x_gc vs y ──────────────────────────────')
print(f'  Pearson  r={pr:+.4f}  p={pp:.4e}')
print(f'  Spearman r={sr:+.4f}  p={sp:.4e}')
print('  (Negative r: larger GC diff → less likely to infect — expected direction ✓)')

sanity = feat_df.groupby('y')['x_gc'].agg(['mean','std','count'])
sanity.index = ['y=0 (no infection)','y=1 (infects)']
print('\n── Sanity: mean x_gc by outcome ────────────────────────')
print(sanity.to_string())
print(f'\n  Interpretation: y=1 pairs have smaller mean x_gc ({sanity.loc["y=1 (infects)","mean"]:.3f}) than y=0 ({sanity.loc["y=0 (no infection)","mean"]:.3f}) ✓')
print('  Signal is weak (p=0.31) — expected; all 6 factors needed for regression.')

group_colors = {'phage':'#E07B54','host':'#4C9A6E','neg':'#5B7EC9'}
group_labels  = {'phage':'Phage','host':'Xanthomonas host','neg':'Cross-genus neg'}

fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Factor 1 — GC Content Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
for kind,grp in gc_df.groupby('kind'):
    grp['gc_whole'].plot.kde(ax=ax, label=group_labels.get(kind,kind),
                             color=group_colors.get(kind,'grey'), lw=2)
    ax.axvline(grp['gc_whole'].mean(), color=group_colors.get(kind,'grey'),
               linestyle='--', alpha=0.5, lw=1)
ax.set_xlabel('Whole-genome GC fraction'); ax.set_ylabel('Density')
ax.set_title('GC% distribution by group\nGC% 分布（依群組）'); ax.legend(fontsize=8)

ax = axes[1]
rng = np.random.default_rng(42)
for kind,grp in feat_df.groupby('host_kind'):
    jy = grp['y'] + rng.uniform(-0.05,0.05,len(grp))
    ax.scatter(grp['x_gc'], jy, label=group_labels.get(kind,kind),
               color=group_colors.get(kind,'grey'), alpha=0.75, edgecolors='white', linewidths=0.5, s=70)
ax.set_xlabel('|GC_phage − GC_host|  (x_gc)'); ax.set_ylabel('y  (1=infects, 0=no)')
ax.set_title('x_gc vs infection label\nx_gc 與感染標籤'); ax.set_yticks([0,1]); ax.legend(fontsize=8)

ax = axes[2]
feat_df['Outcome'] = feat_df['y'].map({0:'No infection (y=0)',1:'Infects (y=1)'})
sns.boxplot(data=feat_df, x='Outcome', y='x_gc', ax=ax,
            palette={'No infection (y=0)':'#5B7EC9','Infects (y=1)':'#4C9A6E'}, width=0.4)
sns.stripplot(data=feat_df, x='Outcome', y='x_gc', ax=ax,
              color='black', alpha=0.45, size=5, jitter=True)
ax.set_xlabel('Outcome'); ax.set_ylabel('|GC_phage − GC_host|')
ax.set_title('GC difference by outcome\n各感染結果之 GC 差異')

plt.tight_layout()
fig.savefig(OUT_DIR / 'gc_density.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✓ Saved gc_density.png')

── Correlation: x_gc vs y ──────────────────────────────
  Pearson  r=-0.1459  p=3.1211e-01
  Spearman r=-0.1594  p=2.6891e-01
  (Negative r: larger GC diff → less likely to infect — expected direction ✓)

── Sanity: mean x_gc by outcome ────────────────────────
                        mean       std  count
y=0 (no infection)  0.107960  0.073818     45
y=1 (infects)       0.073106  0.054047      5

  Interpretation: y=1 pairs have smaller mean x_gc (0.073) than y=0 (0.108) ✓
  Signal is weak (p=0.31) — expected; all 6 factors needed for regression.

✓ Saved gc_density.png


## Results & interpretation / 結果解讀

### Key numbers from this trial run / 本次試跑關鍵數值

| Metric | Value |
|--------|-------|
| Pairs computed | 50 (5 phages × 10 hosts) |
| Mean x_gc (y=1, infects) | **0.073** |
| Mean x_gc (y=0, no infection) | **0.108** |
| Pearson r | **−0.146** (correct direction) |
| Spearman r | **−0.159** (correct direction) |
| p-value | 0.31 (not significant at n=50) |

### Observations / 觀察

- **Xanthomonas hosts cluster at 63–65% GC.** Cross-genus negatives span 33% (S. aureus) to 67% (M. smegmatis), creating large `x_gc` that are easy to separate.
- **黃單胞菌宿主 GC% 集中在 63–65%**，跨屬陰性對照範圍寬（33–67%），大幅拉開 `x_gc` 差距，有助於分類器區分陰性。
- **Signal direction is correct** (infecting pairs have smaller GC difference) but **p-value is non-significant** at n=50. This is expected — GC alone is a coarse proxy. With the full dataset (777 phages × all hosts) the signal should strengthen.
- **信號方向正確**（感染配對的 GC 差值較小），但 n=50 時 p 值不顯著。這是預期的——GC 含量只是粗略代理指標，使用完整數據集（777 個噬菌體）後信號應更顯著。
- **Phage AB720064.1 has 67% GC**, very similar to Pseudomonas and Mycobacterium, which could cause false positives in a GC-only model. The other five factors are needed to disambiguate.

---

## Outputs written / 已寫出的輸出

| File | Description |
|------|-------------|
| `per_genome_gc.csv` | Per-genome GC fraction and length / 每個基因組的 GC 比例與長度 |
| `f01_gc_content.csv` | Per-pair `x_gc` and `x_gc_zscore` — input to regression / 每對配對的特徵值，迴歸模型輸入 |
| `gc_density.png` | Distribution · scatter · box plots / 分布圖、散佈圖與箱型圖 |

---

## Scaling to full dataset / 擴展至完整數據集

1. Update `TRUTH_TABLE_PATH` in Cell 2 to point to `combined_matrix.csv` (after reshaping to long format with columns `phage_id, host_id, y`).
2. Run pharokka on remaining phage genomes so their `pharokka.gbk` appears in `02_annotation/outputs/pharokka_runs/`.
3. Re-run this notebook — no code changes needed.

更換真值表 → 重跑 pharokka → 重新執行筆記本。無需修改程式碼。